In [ ]:
# Config: only NODE_NAME differs (A or B). Keep the slugs identical in both notebooks.
NODE_NAME = "A"
SLUG_A = "dsh-runner-a"   # notebook A ka title/slug
SLUG_B = "dsh-runner-b"   # notebook B ka title/slug


In [ ]:
import os, sys, re, io, json, time, base64, hashlib, shutil, signal, tarfile, threading, subprocess, urllib.request, urllib.error, urllib.parse
from pathlib import Path

START = time.time()


def log(msg):
    print(f"[{NODE_NAME} +{int((time.time() - START) / 60):03d}m] {msg}", flush=True)


def secret(name, default=None):
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name) or default


def num(name, default):
    v = secret(name)
    try:
        return float(v) if v else default
    except ValueError:
        return default


assert NODE_NAME in ("A", "B"), "NODE_NAME must be 'A' or 'B'"
PEER_NAME = "B" if NODE_NAME == "A" else "A"
PEER_SLUG = SLUG_B if NODE_NAME == "A" else SLUG_A

DATABASE_URL = secret("DATABASE_URL")
KUSER = secret("KAGGLE_USERNAME")
KKEY = secret("KAGGLE_KEY")
PASSPHRASE = secret("BACKUP_PASSPHRASE")
_missing = [n for n, v in (("DATABASE_URL", DATABASE_URL), ("KAGGLE_USERNAME", KUSER), ("KAGGLE_KEY", KKEY), ("BACKUP_PASSPHRASE", PASSPHRASE)) if not v]
if _missing:
    raise SystemExit("Missing Kaggle Secrets: " + ", ".join(_missing))

TG_TOKEN = secret("TG_TOKEN")
TG_CHAT_ID = secret("TG_CHAT_ID")
TG_API_BASE = os.environ.get("TG_API_BASE", "https://api.telegram.org")


def notify(text):
    if not (TG_TOKEN and TG_CHAT_ID):
        return False
    data = urllib.parse.urlencode({"chat_id": TG_CHAT_ID, "text": str(text)[:3900], "disable_web_page_preview": "true"}).encode()
    for _ in range(3):
        try:
            urllib.request.urlopen(urllib.request.Request(f"{TG_API_BASE}/bot{TG_TOKEN}/sendMessage", data=data), timeout=20)
            return True
        except Exception:
            time.sleep(2)
    log("telegram notify failed")
    return False


DSH_VERSION = secret("DSH_VERSION", "0.2.2-rc.5")
PORT = 3080
HANDOFF_AFTER = num("HANDOFF_AFTER_H", 9.5) * 3600
HARD_EXIT = num("HARD_EXIT_H", 11.75) * 3600
LEASE_TTL = 90
LEASE_WAIT_MAX = 3600
MAX_BACKUP_MB = num("MAX_BACKUP_MB", 60)
BACKUP_EVERY = 120

_root = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp")
BASE = _root / "dsh_runner"
NODE_DIR = BASE / "node"
NPM_PREFIX = BASE / "npm"
BIN_DIR = BASE / "bin"
DSH_HOME = BASE / "dsh_home"
WORKSPACE = BASE / "workspace"
for _d in (BASE, BIN_DIR, DSH_HOME, WORKSPACE):
    _d.mkdir(parents=True, exist_ok=True)

psycopg2 = None
Fernet = None
KAGGLE = None

SCHEMA = """
create table if not exists runtime_lease(
  id int primary key, holder text, expires_at timestamptz not null default now(),
  waiter text, waiter_at timestamptz);
insert into runtime_lease(id, holder, expires_at) values (1, null, now() - interval '1 second') on conflict (id) do nothing;
create table if not exists runtime_info(
  id int primary key, holder text, launch_url text, public_url text, status text,
  started_at timestamptz, heartbeat_at timestamptz);
insert into runtime_info(id, status) values (1, 'offline') on conflict (id) do nothing;
alter table runtime_info add column if not exists shutdown_at timestamptz;
alter table runtime_info add column if not exists handoff_at timestamptz;
create table if not exists harness_backup_meta(
  name text primary key, parts int not null, size bigint not null, sig text,
  updated_at timestamptz not null default now());
create table if not exists harness_backup(
  name text not null, part int not null, data bytea not null, primary key (name, part));
alter table runtime_lease enable row level security;
alter table runtime_info enable row level security;
alter table harness_backup_meta enable row level security;
alter table harness_backup enable row level security;

create table if not exists launch_state(
  id int primary key, enc text, updated_at timestamptz not null default now());
alter table launch_state enable row level security;
do $$
begin
  if exists (select 1 from pg_roles where rolname = 'anon') then
    execute 'grant select on launch_state to anon';
    if not exists (select 1 from pg_policies
                   where schemaname = 'public' and tablename = 'launch_state' and policyname = 'dsh_link_public_read') then
      execute 'create policy dsh_link_public_read on launch_state for select to anon using (true)';
    end if;
  end if;
end $$;
notify pgrst, 'reload schema';
"""

# ---------------------------------------------------------------- db helpers

def db(sql, params=None, fetch=False, tries=5):
    delay = 1
    for i in range(tries):
        conn = None
        try:
            conn = psycopg2.connect(DATABASE_URL, connect_timeout=15)
            conn.autocommit = True
            with conn.cursor() as cur:
                cur.execute(sql, params)
                return cur.fetchall() if fetch else None
        except Exception as e:
            if i == tries - 1:
                raise
            log(f"db retry {i + 1}: {type(e).__name__}: {str(e)[:120]}")
            time.sleep(delay)
            delay = min(delay * 2, 15)
        finally:
            if conn is not None:
                conn.close()


def try_acquire():
    r = db(
        "insert into runtime_lease(id, holder, expires_at) values (1, %s, now() + make_interval(secs => %s)) "
        "on conflict (id) do update set holder = excluded.holder, expires_at = excluded.expires_at, waiter = null, waiter_at = null "
        "where runtime_lease.expires_at < now() or runtime_lease.holder = excluded.holder returning holder",
        (NODE_NAME, LEASE_TTL), fetch=True)
    return bool(r)


def mark_waiter():
    db("update runtime_lease set waiter = %s, waiter_at = now() where id = 1", (NODE_NAME,))


def peer_ready():
    r = db("select 1 from runtime_lease where id = 1 and waiter = %s and waiter_at > now() - interval '60 seconds'", (PEER_NAME,), fetch=True)
    return bool(r)


def release_lease():
    db("update runtime_lease set holder = null, expires_at = now() - interval '1 second' where id = 1 and holder = %s", (NODE_NAME,))


def set_status(status):
    db("update runtime_info set status = %s, heartbeat_at = now() where id = 1 and holder = %s", (status, NODE_NAME))


def publish(status):
    db("update runtime_info set holder = %s, launch_url = %s, public_url = %s, status = %s, started_at = now(), heartbeat_at = now() where id = 1",
       (NODE_NAME, S.launch_url, f"https://{S.host}/", status))
    publish_remote(status)


lost = threading.Event()
stop_renew = threading.Event()


def renewer():
    last_ok = time.time()
    while not stop_renew.wait(30):
        try:
            r = db("update runtime_lease set expires_at = now() + make_interval(secs => %s) where id = 1 and holder = %s returning 1",
                   (LEASE_TTL, NODE_NAME), fetch=True, tries=2)
            if not r:
                log("lease lost (another holder)")
                lost.set()
                return
            last_ok = time.time()
            db("update runtime_info set heartbeat_at = now() where id = 1 and holder = %s", (NODE_NAME,), tries=1)
            link_heartbeat()
        except Exception as e:
            log(f"renew error: {type(e).__name__}: {str(e)[:120]}")
            if time.time() - last_ok > LEASE_TTL - 15:
                log("lease considered lost (db unreachable)")
                lost.set()
                return

# ------------------------------------------------------------ backup / restore

EXCLUDE_DIRS = {"node_modules", "__pycache__", ".cache"}
CHUNK = 4 * 1024 * 1024
LAST_SIG = {}


def iter_files(root):
    for dp, dn, fn in os.walk(root):
        dn[:] = [d for d in dn if d not in EXCLUDE_DIRS]
        for f in fn:
            if f.endswith(".log") or f.endswith(".sock"):
                continue
            p = Path(dp) / f
            if p.is_symlink() or not p.is_file():
                continue
            yield p


def sig_of(root):
    h = hashlib.sha256()
    for p in sorted(iter_files(root)):
        try:
            st = p.stat()
        except OSError:
            continue
        h.update(f"{p.relative_to(root)}|{st.st_size}|{int(st.st_mtime)}\n".encode())
    return h.hexdigest()


def _key(salt):
    return base64.urlsafe_b64encode(hashlib.scrypt(PASSPHRASE.encode(), salt=salt, n=2 ** 14, r=8, p=1, dklen=32))


def encrypt(raw):
    salt = os.urandom(16)
    return b"DSH1" + salt + Fernet(_key(salt)).encrypt(raw)


def decrypt(blob):
    if blob[:4] != b"DSH1":
        raise ValueError("backup header invalid")
    return Fernet(_key(blob[4:20])).decrypt(blob[20:])


def make_blob(root):
    buf = io.BytesIO()
    with tarfile.open(fileobj=buf, mode="w:gz") as t:
        for p in iter_files(root):
            try:
                t.add(p, arcname=str(p.relative_to(root)), recursive=False)
            except FileNotFoundError:
                pass
    raw = buf.getvalue()
    with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as t:
        t.getmembers()
    return encrypt(raw)


def save_backup(name, blob, sig):
    parts = [blob[i:i + CHUNK] for i in range(0, len(blob), CHUNK)] or [b""]
    for attempt in range(3):
        conn = None
        try:
            conn = psycopg2.connect(DATABASE_URL, connect_timeout=15)
            with conn:
                with conn.cursor() as cur:
                    cur.execute("delete from harness_backup where name = %s", (name,))
                    for i, part in enumerate(parts):
                        cur.execute("insert into harness_backup(name, part, data) values (%s, %s, %s)", (name, i, psycopg2.Binary(part)))
                    cur.execute(
                        "insert into harness_backup_meta(name, parts, size, sig, updated_at) values (%s, %s, %s, %s, now()) "
                        "on conflict (name) do update set parts = excluded.parts, size = excluded.size, sig = excluded.sig, updated_at = now()",
                        (name, len(parts), len(blob), sig))
            return True
        except Exception as e:
            log(f"backup save retry {attempt + 1}: {type(e).__name__}: {str(e)[:120]}")
            time.sleep(5 * (attempt + 1))
        finally:
            if conn is not None:
                conn.close()
    return False


def load_backup(name):
    meta = db("select parts from harness_backup_meta where name = %s", (name,), fetch=True)
    if not meta:
        return None
    rows = db("select data from harness_backup where name = %s order by part", (name,), fetch=True)
    if len(rows) != meta[0][0]:
        raise RuntimeError(f"backup '{name}' incomplete ({len(rows)}/{meta[0][0]} parts)")
    return b"".join(bytes(r[0]) for r in rows)


def restore(name, root):
    blob = load_backup(name)
    if blob is None:
        log(f"no backup for '{name}' (fresh start)")
        return
    raw = decrypt(blob)
    root.mkdir(parents=True, exist_ok=True)
    rr = root.resolve()
    n = 0
    with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as t:
        for m in t.getmembers():
            if not m.isfile():
                continue
            dest = (root / m.name).resolve()
            if rr not in dest.parents:
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            with t.extractfile(m) as src, open(dest, "wb") as out:
                shutil.copyfileobj(src, out)
            os.chmod(dest, m.mode & 0o777)
            os.utime(dest, (m.mtime, m.mtime))
            n += 1
    log(f"restored '{name}': {n} files")


def backup_all():
    for name, root in (("dsh_home", DSH_HOME), ("workspace", WORKSPACE)):
        try:
            s = sig_of(root)
            if s == LAST_SIG.get(name):
                continue
            blob = make_blob(root)
            if len(blob) > MAX_BACKUP_MB * 1024 * 1024:
                log(f"backup '{name}' skipped: {len(blob) // 1048576} MB > limit {MAX_BACKUP_MB} MB")
                continue
            if save_backup(name, blob, s):
                LAST_SIG[name] = s
                log(f"backup '{name}' saved ({len(blob) // 1024} KB)")
        except Exception as e:
            log(f"backup '{name}' failed: {type(e).__name__}: {str(e)[:160]}")

# ------------------------------------------------------------------- installs

def download(url, dest):
    req = urllib.request.Request(url, headers={"User-Agent": "dsh-runner"})
    with urllib.request.urlopen(req, timeout=180) as r, open(dest, "wb") as f:
        shutil.copyfileobj(r, f)


def tool_env():
    env = os.environ.copy()
    env["PATH"] = os.pathsep.join([str(NODE_DIR / "bin"), str(NPM_PREFIX / "bin"), str(BIN_DIR), env.get("PATH", "")])
    env["DSH_HOME"] = str(DSH_HOME)
    for k in list(env):
        if k.startswith("KAGGLE_") or k in ("DATABASE_URL", "BACKUP_PASSPHRASE"):
            env.pop(k, None)
    return env


def kenv():
    e = os.environ.copy()
    e["KAGGLE_USERNAME"] = KUSER
    e["KAGGLE_KEY"] = KKEY
    return e


def install_node():
    if (NODE_DIR / "bin" / "node").exists():
        return
    base = "https://nodejs.org/dist/latest-v24.x/"
    sums = urllib.request.urlopen(base + "SHASUMS256.txt", timeout=60).read().decode()
    sha, fname = next(l.split() for l in sums.splitlines() if l.endswith("linux-x64.tar.xz"))
    tmp = BASE / fname
    download(base + fname, tmp)
    if hashlib.sha256(tmp.read_bytes()).hexdigest() != sha:
        raise RuntimeError("node checksum mismatch")
    tmpdir = BASE / "node_tmp"
    shutil.rmtree(tmpdir, ignore_errors=True)
    with tarfile.open(tmp, "r:xz") as t:
        t.extractall(tmpdir)
    shutil.rmtree(NODE_DIR, ignore_errors=True)
    shutil.move(str(next(tmpdir.iterdir())), str(NODE_DIR))
    shutil.rmtree(tmpdir, ignore_errors=True)
    tmp.unlink()


def install_dsh():
    if (NPM_PREFIX / "bin" / "dsh").exists():
        return
    subprocess.run([str(NODE_DIR / "bin" / "npm"), "install", "-g", "--prefix", str(NPM_PREFIX), "--no-audit", "--no-fund",
                    "--loglevel=error", f"@deepseek-ai/dsh@{DSH_VERSION}"], check=True, env=tool_env())


def install_cloudflared():
    dest = BIN_DIR / "cloudflared"
    if dest.exists():
        return
    download("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", dest)
    os.chmod(dest, 0o755)


def setup():
    global psycopg2, Fernet, KAGGLE
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "psycopg2-binary", "cryptography", "kaggle"], check=True)
    import psycopg2 as _p
    from cryptography.fernet import Fernet as _F
    psycopg2, Fernet = _p, _F
    exe = shutil.which("kaggle", path=os.path.dirname(sys.executable) + os.pathsep + os.environ.get("PATH", ""))
    KAGGLE = [exe] if exe else [sys.executable, "-m", "kaggle.cli"]
    db(SCHEMA)
    install_node()
    install_dsh()
    install_cloudflared()
    log("node " + subprocess.run([str(NODE_DIR / "bin" / "node"), "--version"], capture_output=True, text=True).stdout.strip() + f", dsh {DSH_VERSION} ready")

# ---------------------------------------------------------------- processes

TOKEN_RX = re.compile(r"token=[^\s&]+")


def kill(p, grace=15):
    if p is None or p.poll() is not None:
        return
    try:
        os.killpg(p.pid, signal.SIGINT)
    except ProcessLookupError:
        return
    try:
        p.wait(grace)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(p.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        p.wait(5)


def spawn(tag, cmd, pattern, timeout, env=None, cwd=None, quiet=False):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env=env, cwd=cwd, start_new_session=True)
    rx = re.compile(pattern)
    found = {}
    ev = threading.Event()

    def pump():
        for line in p.stdout:
            line = line.rstrip()
            hit = False
            if not ev.is_set():
                m = rx.search(line)
                if m:
                    found["m"] = m.group(1)
                    hit = True
                    ev.set()
            if not quiet or "ERR" in line or hit:
                log(f"{tag}: {TOKEN_RX.sub('token=***', line)}")

    threading.Thread(target=pump, daemon=True).start()
    end = time.time() + timeout
    while time.time() < end and not ev.is_set() and p.poll() is None:
        time.sleep(0.5)
    if not ev.is_set():
        kill(p, 5)
        raise RuntimeError(f"{tag}: did not become ready (exit={p.poll()})")
    return p, found["m"]



# ------------------------------------------------------------------ auth shim
# dsh web launch token ko SameSite=Strict cookie me badalta hai (303 redirect ke beech). Kisi doosre app / in-app
# browser / launcher page se khulne par wo cookie kho jati hai -> "authentication required". Ye loopback proxy token URL
# ko redirect ke bina seedha index + cookie ke saath answer karta hai aur baaki har request me session cookie khud lagata hai.
import contextlib, hmac, http.client, secrets, socket, urllib.parse


class AuthShim:
    INDEX = ("/", "/index.html")

    def __init__(self, upstream_port, host="127.0.0.1"):
        self.up = upstream_port
        self.host = host
        self._raw = None
        self._token = None
        self._cookies = {}
        self._sk = secrets.token_urlsafe(32)   # shim ki apni cookie ka secret (sirf token-holder ko milta hai)
        self._lock = threading.Lock()
        self._stop = threading.Event()
        self._ls = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self._ls.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        self._ls.bind((host, 0))
        self._ls.listen(64)
        self.port = self._ls.getsockname()[1]

    def start(self):
        threading.Thread(target=self._accept, daemon=True).start()

    def arm(self, raw_token, public_host):
        self._raw = raw_token
        self._token = urllib.parse.unquote(raw_token)
        ok = self._cookie(public_host) is not None
        log("auth shim: session cookie " + ("ready" if ok else "NOT minted (token exchange failed)"))

    def stop(self):
        self._stop.set()
        with contextlib.suppress(OSError):
            self._ls.shutdown(socket.SHUT_RDWR)
        with contextlib.suppress(OSError):
            self._ls.close()

    def _accept(self):
        while not self._stop.is_set():
            try:
                c, _ = self._ls.accept()
            except OSError:
                return
            threading.Thread(target=self._serve, args=(c,), daemon=True).start()

    @staticmethod
    def _read_head(c):
        buf = b""
        while b"\r\n\r\n" not in buf:
            d = c.recv(65536)
            if not d:
                return None, b""
            buf += d
            if len(buf) > 65536:
                raise OSError("request head too large")
        head, _, rest = buf.partition(b"\r\n\r\n")
        return head.decode("latin-1"), rest

    @staticmethod
    def _pump(a, b):
        try:
            while True:
                d = a.recv(65536)
                if not d:
                    break
                b.sendall(d)
        except OSError:
            pass
        finally:
            with contextlib.suppress(OSError):
                b.shutdown(socket.SHUT_WR)

    SKN = "dsh-shim"

    def _shim_ok(self, cookie_header):
        for seg in (cookie_header or "").split(";"):
            k, _, v = seg.strip().partition("=")
            if k == self.SKN:
                return hmac.compare_digest(v.encode(), self._sk.encode())
        return False

    def _token_ok(self, query):
        if not self._token:
            return False
        v = urllib.parse.parse_qs(query).get("token", [])
        return len(v) == 1 and hmac.compare_digest(v[0].encode(), self._token.encode())

    def _cookie(self, host):
        with self._lock:
            if host in self._cookies or not self._raw:
                return self._cookies.get(host)
            cn = http.client.HTTPConnection(self.host, self.up, timeout=30)
            try:
                cn.request("GET", f"/?token={self._raw}", headers={"Host": host})
                r = cn.getresponse()
                r.read()
                ck = r.getheader("set-cookie")
                if r.status != 303 or not ck:
                    log(f"auth shim: token exchange answered HTTP {r.status}")
                    return None
                self._cookies[host] = ck
                return ck
            except OSError as e:
                log(f"auth shim: token exchange failed: {e}")
                return None
            finally:
                cn.close()

    def _index(self, c, method, host):
        ck = self._cookie(host)
        if not ck:
            return False
        cn = http.client.HTTPConnection(self.host, self.up, timeout=30)
        try:
            cn.request("GET", "/", headers={"Host": host, "Cookie": ck.split(";", 1)[0].strip()})
            r = cn.getresponse()
            body = r.read()
            if r.status != 200:
                return False
            ctype = r.getheader("content-type") or "text/html; charset=utf-8"
        except OSError:
            return False
        finally:
            cn.close()
        head = ["HTTP/1.1 200 OK", f"Content-Type: {ctype}", f"Content-Length: {len(body)}",
                "Cache-Control: no-store", "Referrer-Policy: no-referrer", f"Set-Cookie: {ck}",
                f"Set-Cookie: {self.SKN}={self._sk}; Path=/; Max-Age=2592000; HttpOnly; Secure; SameSite=Lax",
                "Connection: close"]
        c.sendall(("\r\n".join(head) + "\r\n\r\n").encode("latin-1"))
        if method != "HEAD":
            c.sendall(body)
        return True

    def _serve(self, c):
        up = None
        try:
            c.settimeout(30)
            head, rest = self._read_head(c)
            if head is None:
                return
            lines = head.split("\r\n")
            method, target, ver = lines[0].split(" ", 2)
            hdrs = []
            for ln in lines[1:]:
                k, _, v = ln.partition(":")
                if k.strip():
                    hdrs.append((k.strip(), v.strip()))

            def hget(n):
                return next((v for k, v in hdrs if k.lower() == n), None)

            host = hget("host") or ""
            sp = urllib.parse.urlsplit(target)
            has_tok = "token=" in sp.query
            shim_ok = self._shim_ok(hget("cookie"))
            if sp.path in self.INDEX:
                served = False
                if method in ("GET", "HEAD") and ((has_tok and self._token_ok(sp.query)) or (not has_tok and shim_ok)):
                    served = self._index(c, method, host)
                log(f"auth shim: index {method} token={'y' if has_tok else 'n'} shimck={'y' if shim_ok else 'n'} "
                    f"sfs={hget('sec-fetch-site') or '-'} -> {'served' if served else 'forwarded'}")
                if served:
                    return
            c.settimeout(None)
            upgrade = "upgrade" in (hget("connection") or "").lower() or hget("upgrade") is not None
            out = [(k, v) for k, v in hdrs if k.lower() not in ("connection", "keep-alive", "proxy-connection")]
            if sp.path not in self.INDEX and shim_ok:
                ck = self._cookie(host)
                if ck:
                    old = hget("cookie")
                    out = [(k, v) for k, v in out if k.lower() != "cookie"]
                    out.append(("Cookie", ck.split(";", 1)[0].strip() + ("; " + old if old else "")))
            out.append(("Connection", "Upgrade" if upgrade else "close"))
            up = socket.create_connection((self.host, self.up), timeout=30)
            up.settimeout(None)
            up.sendall((f"{method} {target} {ver}\r\n" + "".join(f"{k}: {v}\r\n" for k, v in out) + "\r\n").encode("latin-1") + rest)
            helper = threading.Thread(target=self._pump, args=(up, c), daemon=True)
            helper.start()
            self._pump(c, up)
            helper.join(5)
        except (OSError, ValueError):
            pass
        finally:
            for s in (up, c):
                if s is not None:
                    with contextlib.suppress(OSError):
                        s.close()


class Stack:
    shim = None
    tunnel = None
    dsh = None
    host = None
    launch_url = None


S = Stack()


def start_stack():
    S.shim = AuthShim(PORT)
    S.shim.start()
    try:
        t, url = spawn("cf", [str(BIN_DIR / "cloudflared"), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{S.shim.port}"],
                       r"(https://[a-z0-9-]+\.trycloudflare\.com)", 90, env=tool_env(), quiet=True)
    except Exception:
        S.shim.stop()
        S.shim = None
        raise
    S.tunnel = t
    S.host = url.split("//", 1)[1]
    try:
        d, printed = spawn("dsh", [str(NPM_PREFIX / "bin" / "dsh"), "--profile", "web", "--no-open", "--port", str(PORT),
                                   "--public-url", f"https://{S.host}/", "--trusted-host", S.host],
                           r"dsh web:\s*(\S+)", 120, env=tool_env(), cwd=str(WORKSPACE))
    except Exception:
        kill(S.tunnel, 5)
        S.shim.stop()
        S.shim = None
        raise
    S.dsh = d
    tok = re.search(r"token=([^\s&]+)", printed)
    if not tok:
        kill(S.dsh, 5)
        kill(S.tunnel, 5)
        S.shim.stop()
        S.shim = None
        raise RuntimeError("launch token not found in dsh output")
    S.launch_url = f"https://{S.host}/?token={tok.group(1)}"
    S.shim.arm(tok.group(1), S.host)
    log(f"stack up: https://{S.host}/")


def stop_stack():
    kill(S.dsh)
    kill(S.tunnel)
    if S.shim is not None:
        S.shim.stop()
        S.shim = None
    S.dsh = None
    S.tunnel = None


def http_alive(url, timeout=10, dead_codes=()):
    try:
        urllib.request.urlopen(url, timeout=timeout)
        return True
    except urllib.error.HTTPError as e:
        return e.code not in dead_codes
    except Exception:
        return False


TUNNEL_DEAD = (502, 503, 504, 530)


def wait_public(seconds=90):
    end = time.time() + seconds
    while time.time() < end:
        if http_alive(f"https://{S.host}/", 10, TUNNEL_DEAD):
            return True
        time.sleep(3)
    log("warning: tunnel URL not reachable yet, publishing anyway")
    return False

G = {"start_db": None, "reason": None}
stop_poll = threading.Event()


def shutdown_flag():
    r = db("select 1 from runtime_info where id = 1 and shutdown_at is not null and (shutdown_at > %s or "
           "(handoff_at is not null and shutdown_at > handoff_at and handoff_at > %s - interval '2 hours'))",
           (G["start_db"], G["start_db"]), fetch=True, tries=2)
    return bool(r)


def tg_poller():
    if not (TG_TOKEN and TG_CHAT_ID):
        return
    offset = None
    fails = 0
    while not stop_poll.is_set():
        try:
            params = {"timeout": 25, "allowed_updates": json.dumps(["message"])}
            if offset is not None:
                params["offset"] = offset
            with urllib.request.urlopen(f"{TG_API_BASE}/bot{TG_TOKEN}/getUpdates?" + urllib.parse.urlencode(params), timeout=40) as r:
                data = json.load(r)
            fails = 0
            for u in data.get("result", []):
                offset = u["update_id"] + 1
                m = u.get("message") or {}
                if str((m.get("chat") or {}).get("id")) != str(TG_CHAT_ID):
                    continue
                if (m.get("text") or "").strip().lower().startswith("/shutdown") and time.time() - m.get("date", 0) < 600:
                    db("update runtime_info set shutdown_at = now() where id = 1")
                    log("/shutdown received")
                    notify("🛑 /shutdown received. Both notebooks are shutting down.")
        except urllib.error.HTTPError as e:
            if e.code == 409:
                log("telegram 409: bot token is used by another program, /shutdown disabled")
                notify("⚠️ /shutdown will not work: this bot token is also used by another program (409). Use a separate bot token (a new one from BotFather) for this notebook.")
                return
            time.sleep(5)
        except Exception:
            fails += 1
            time.sleep(min(5 * fails, 30))


def announce(note=""):
    notify(f"✅ Harness online [{NODE_NAME}] {note}\n{S.launch_url}\n(secret link, do not share)")

# ------------------------------------------------------------------ handoff

INJECT_KEYS = ["DATABASE_URL", "KAGGLE_USERNAME", "KAGGLE_KEY", "BACKUP_PASSPHRASE", "TG_TOKEN", "TG_CHAT_ID",
               "DSH_VERSION", "HANDOFF_AFTER_H", "HARD_EXIT_H", "MAX_BACKUP_MB",
               "LINK_PASSPHRASE", "LINK_DB", "LINK_TABLE", "LINK_API_URL", "LINK_API_SECRET",
               "LINK_REPO", "LINK_PAT", "LINK_BRANCH", "LINK_PATH", "LINK_SALT"]


def inject_secrets(d):
    f = next(d.glob("*.ipynb"))
    nb = json.loads(f.read_text(encoding="utf-8"))
    nb["cells"] = [c for c in nb["cells"] if "#__injected__" not in "".join(c.get("source", []))]
    vals = {k: secret(k) for k in INJECT_KEYS if secret(k)}
    src = "#__injected__\nimport os\nos.environ.update(" + repr(vals) + ")\n"
    nb["cells"].insert(0, {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [],
                           "source": src.splitlines(True)})
    f.write_text(json.dumps(nb), encoding="utf-8")


def peer_status(slug):
    st = subprocess.run(KAGGLE + ["kernels", "status", slug], env=kenv(), capture_output=True, text=True, timeout=120)
    return (st.stdout + st.stderr).strip()


def push_peer(slug):
    d = BASE / "peer"
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True)
    r = subprocess.run(KAGGLE + ["kernels", "pull", slug, "-p", str(d), "-m"], env=kenv(), capture_output=True, text=True, timeout=300)
    if r.returncode != 0:
        raise RuntimeError("pull failed: " + (r.stdout + r.stderr)[-300:])
    inject_secrets(d)
    r = subprocess.run(KAGGLE + ["kernels", "push", "-p", str(d)], env=kenv(), capture_output=True, text=True, timeout=300)
    if r.returncode != 0:
        raise RuntimeError("push failed: " + (r.stdout + r.stderr)[-300:])
    log("peer pushed: " + (r.stdout.strip() or "ok")[-200:])


def trigger_peer():
    slug = f"{KUSER}/{PEER_SLUG}"
    for attempt in range(1, 6):
        try:
            if shutdown_flag():
                log("shutdown requested, not pushing peer")
                return True
            out = peer_status(slug)
            log(f"peer status: {out[-150:]}")
            up = out.upper()
            if "RUNNING" in up or "QUEUED" in up:
                log("peer already running/queued, not pushing")
            else:
                push_peer(slug)
            end = time.time() + 300
            while time.time() < end:
                if peer_ready():
                    return True
                s = peer_status(slug).upper()
                if "ERROR" in s or "CANCEL" in s or "COMPLETE" in s:
                    notify(f"\u26a0\ufe0f [{NODE_NAME}] peer {PEER_NAME} status: {s[-120:]} \u2014 re-pushing (attempt {attempt}).")
                    break
                time.sleep(20)
            else:
                notify(f"\u2139\ufe0f [{NODE_NAME}] peer {PEER_NAME} still starting/queued: {peer_status(slug)[-120:]}")
                return True
        except Exception as e:
            log(f"trigger_peer attempt {attempt}: {type(e).__name__}: {str(e)[:300]}")
            time.sleep(30 * attempt)
    notify(f"\u26a0\ufe0f [{NODE_NAME}] peer {PEER_NAME} failed to start. The active notebook will run for ~{int(HARD_EXIT / 3600)}h; after that, manually Save & Run All {PEER_NAME}.")
    return False


H = {"started": False, "ok": None}


def _handoff_job():
    H["ok"] = trigger_peer()

# --------------------------------------------------------------------- main

def seed_resume_files():
    a = WORKSPACE / "AGENTS.md"
    if not a.exists():
        a.write_text(
            "# Runtime notes\n"
            "- This runtime restarts/hands over every ~11h; a running task can be killed mid-step at any time.\n"
            "- For any multi-step task: first write the plan to PROGRESS.md, then update it after EVERY step "
            "(Done / Next / key file paths / pending commands).\n"
            "- At session start, or when told to continue: read PROGRESS.md and resume from 'Next'. Do not restart finished steps.\n"
            "- When the task is fully finished, set the Status line in PROGRESS.md to DONE.\n", encoding="utf-8")
    p = WORKSPACE / "PROGRESS.md"
    if not p.exists():
        p.write_text("# PROGRESS\nStatus: IDLE\n\n## Done\n\n## Next\n", encoding="utf-8")


SEND_MAX = 49 * 1024 * 1024
SEND_SKIP = {"AGENTS.md", "PROGRESS.md"}
SENT = {}


def _fkey(p):
    st = p.stat()
    return (st.st_size, int(st.st_mtime))


def baseline_sent():
    for p in iter_files(WORKSPACE):
        try:
            SENT[p] = _fkey(p)
        except OSError:
            pass


def send_file(p):
    rel = str(p.relative_to(WORKSPACE))
    name = p.name.replace('"', "_").replace("\r", "_").replace("\n", "_")
    b = "----dsh" + os.urandom(8).hex()
    caption = f"[{NODE_NAME}] {rel}"[:1000]
    head = (f"--{b}\r\nContent-Disposition: form-data; name=\"chat_id\"\r\n\r\n{TG_CHAT_ID}\r\n"
            f"--{b}\r\nContent-Disposition: form-data; name=\"caption\"\r\n\r\n{caption}\r\n"
            f"--{b}\r\nContent-Disposition: form-data; name=\"document\"; filename=\"{name}\"\r\n"
            f"Content-Type: application/octet-stream\r\n\r\n").encode("utf-8")
    for attempt in range(3):
        try:
            body = head + p.read_bytes() + f"\r\n--{b}--\r\n".encode()
            req = urllib.request.Request(f"{TG_API_BASE}/bot{TG_TOKEN}/sendDocument", data=body,
                                         headers={"Content-Type": f"multipart/form-data; boundary={b}"})
            urllib.request.urlopen(req, timeout=120)
            return True
        except Exception as e:
            log(f"send_file {rel} attempt {attempt + 1}: {type(e).__name__}: {str(e)[:120]}")
            time.sleep(3 * (attempt + 1))
    return False


def file_sender():
    if not (TG_TOKEN and TG_CHAT_ID):
        return
    pending = {}
    while not stop_poll.wait(20):
        try:
            for p in iter_files(WORKSPACE):
                rel = p.relative_to(WORKSPACE)
                if p.name in SEND_SKIP or any(x.startswith(".") for x in rel.parts):
                    continue
                try:
                    k = _fkey(p)
                except OSError:
                    continue
                if SENT.get(p) == k:
                    continue
                if pending.get(p) != k:
                    pending[p] = k
                    continue
                pending.pop(p, None)
                if k[0] == 0:
                    SENT[p] = k
                elif k[0] > SEND_MAX:
                    SENT[p] = k
                    notify(f"📎 [{NODE_NAME}] {rel} is too large ({k[0] // 1048576} MB > 49 MB), not sent to Telegram.")
                elif send_file(p):
                    SENT[p] = k
        except Exception as e:
            log(f"file_sender error: {type(e).__name__}: {str(e)[:160]}")


def shutdown():
    stop_renew.set()
    stop_poll.set()
    try:
        stop_stack()
    except Exception as e:
        log(f"stop_stack error: {e}")
    try:
        if not lost.is_set():
            backup_all()
    except Exception as e:
        log(f"final backup error: {e}")
    try:
        db("update runtime_info set status = 'offline', launch_url = null, public_url = null where id = 1 and holder = %s", (NODE_NAME,))
        publish_remote("offline", force=True)
        release_lease()
        log("lease released")
    except Exception as e:
        log(f"release error: {e}")
    if G["reason"] == "shutdown":
        notify(f"🛑 [{NODE_NAME}] stopped (state saved + lease released).")


# ============================================================ DSH LINK PUBLISHER
# Ye block notebook ke publish()/shutdown()/renewer() se call hota hai aur current
# tunnel link (AES-GCM encrypted) ko in targets me se jitne configured hain unme likhta hai:
#
#   1. SUPABASE table  (default ON, koi extra token nahi -- wahi DATABASE_URL connection)
#   2. HTTP endpoint   (optional, e.g. Cloudflare Worker) -- LINK_API_URL + LINK_API_SECRET
#   3. GitHub repo     (optional backup)                   -- LINK_REPO + LINK_PAT
#
# Required Kaggle Secrets: LINK_PASSPHRASE   (optional: LINK_DB=0 se supabase off)
# App inhi me se sabse fresh payload uthaati hai.

LINK_PASSPHRASE = secret("LINK_PASSPHRASE")
LINK_TABLE = secret("LINK_TABLE", "launch_state")
LINK_DB_ON = str(secret("LINK_DB", "1")).strip().lower() not in ("0", "false", "no", "off")
LINK_API_URL = secret("LINK_API_URL")
LINK_API_SECRET = secret("LINK_API_SECRET")
LINK_REPO = secret("LINK_REPO")
LINK_PAT = secret("LINK_PAT")
LINK_BRANCH = secret("LINK_BRANCH", "main")
LINK_PATH = secret("LINK_PATH", "state/current.json")
LINK_SALT = secret("LINK_SALT", "dsh-link-v1")
LINK_ITER = 200000
LINK_MIN_INTERVAL = 480                          # heartbeat re-publish interval (sec)
LINK_GH_INTERVAL = 3600                          # github fallback: itna hi baar commit (repo bloat se bacho)
LINK_API = os.environ.get("GITHUB_API_BASE", "https://api.github.com")

if not re.match(r"^[a-z_][a-z0-9_]*$", LINK_TABLE):     # config se aaya naam, validate kar lo
    raise SystemExit("LINK_TABLE invalid: " + LINK_TABLE)

LINK = {"status": None, "at": 0.0, "warned": False, "err_logged": 0.0, "gh_status": None, "gh_at": 0.0}


def _link_blob(payload):
    """AES-GCM encrypt karke encrypted-envelope JSON return karta hai (browser WebCrypto compatible)."""
    from cryptography.hazmat.primitives.ciphers.aead import AESGCM
    from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
    from cryptography.hazmat.primitives import hashes
    kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=LINK_SALT.encode(), iterations=LINK_ITER)
    key = kdf.derive(LINK_PASSPHRASE.encode())
    iv = os.urandom(12)
    ct = AESGCM(key).encrypt(iv, json.dumps(payload).encode(), None)
    b64 = lambda x: base64.b64encode(x).decode()
    return json.dumps({"v": 1, "alg": "AES-GCM", "kdf": "PBKDF2-SHA256",
                       "it": LINK_ITER, "salt": LINK_SALT, "iv": b64(iv), "ct": b64(ct)}, indent=1)


# ------------------------------------------------------------------ target 1: supabase/postgres
def _link_db_push(blob):
    db(f"insert into {LINK_TABLE}(id, enc, updated_at) values (1, %s, now()) "
       f"on conflict (id) do update set enc = excluded.enc, updated_at = now()", (blob,))
    return True


# ------------------------------------------------------------------ target 2: http (worker)
def _link_api_push(blob):
    req = urllib.request.Request(LINK_API_URL, data=blob.encode(), method="POST")
    req.add_header("Authorization", "Bearer " + LINK_API_SECRET)
    req.add_header("Content-Type", "application/json")
    req.add_header("User-Agent", "dsh-link/1.0")
    with urllib.request.urlopen(req, timeout=25) as r:
        return 200 <= r.status < 300


# ------------------------------------------------------------------ target 3: github
def _link_gh(method, url, body=None):
    req = urllib.request.Request(url, data=body, method=method)
    req.add_header("Authorization", "Bearer " + LINK_PAT)
    req.add_header("Accept", "application/vnd.github+json")
    req.add_header("X-GitHub-Api-Version", "2022-11-28")
    req.add_header("User-Agent", "dsh-link/1.0")
    if body is not None:
        req.add_header("Content-Type", "application/json")
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.load(r)


def _link_gh_push(blob, msg):
    base = f"{LINK_API}/repos/{LINK_REPO}/contents/{LINK_PATH}"
    content = base64.b64encode(blob.encode()).decode()
    for attempt in range(3):
        sha = None
        try:
            sha = _link_gh("GET", f"{base}?ref={urllib.parse.quote(LINK_BRANCH)}").get("sha")
        except urllib.error.HTTPError as e:
            if e.code != 404:
                raise
        body = {"message": msg, "content": content, "branch": LINK_BRANCH}
        if sha:
            body["sha"] = sha
        try:
            _link_gh("PUT", base, json.dumps(body).encode())
            return True
        except urllib.error.HTTPError as e:
            if e.code in (409, 422, 500) and attempt < 2:
                time.sleep(2 + attempt * 3)
                continue
            raise
    return False


def _link_targets():
    t = []
    if LINK_DB_ON and DATABASE_URL:
        t.append("db")
    if LINK_API_URL and LINK_API_SECRET:
        t.append("api")
    if LINK_REPO and LINK_PAT:
        t.append("github")
    return t


def _link_still_mine():
    """Handoff ke baad purana node naye node ka 'online' status overwrite na kare."""
    try:
        return bool(db("select 1 from runtime_info where id = 1 and holder = %s", (NODE_NAME,), fetch=True, tries=2))
    except Exception:
        return True


def publish_remote(status, force=False):
    """Best-effort: link state configured targets me likho. Kabhi exception nahi throw karta."""
    try:
        targets = _link_targets()
        if not LINK_PASSPHRASE or not targets:
            if not LINK["warned"]:
                LINK["warned"] = True
                log("launcher: koi target configured nahi (LINK_PASSPHRASE + LINK_DB ya LINK_API_URL/SECRET ya LINK_REPO/PAT) -> app link publish off")
            return False
        if status != "offline" and not S.launch_url:
            return False
        if status == "offline" and not _link_still_mine():
            log("launcher: peer already took over, offline write skipped")
            return False
        now = time.time()
        if not force and LINK["status"] == status and now - LINK["at"] < LINK_MIN_INTERVAL:
            return False
        payload = {"v": 1, "status": status, "node": NODE_NAME, "ts": int(now)}
        if status != "offline":
            payload["url"] = S.launch_url
            payload["public"] = f"https://{S.host}/" if S.host else None
        blob = _link_blob(payload)
        msg = f"dsh-link: {status} [{NODE_NAME}] {time.strftime('%Y-%m-%d %H:%M:%SZ', time.gmtime(now))}"
        gh_due = force or status != LINK["gh_status"] or (now - LINK["gh_at"]) > LINK_GH_INTERVAL
        done = []
        for t in targets:
            try:
                if t == "db":
                    _link_db_push(blob)
                elif t == "api":
                    _link_api_push(blob)
                else:
                    if not gh_due:
                        continue
                    _link_gh_push(blob, msg)
                    LINK["gh_status"], LINK["gh_at"] = status, now
                done.append(t)
            except Exception as e:
                log(f"launcher[{t}] error: {type(e).__name__}: {str(e)[:160]}")
        if done:
            LINK["status"], LINK["at"] = status, now
            log(f"launcher: published '{status}' -> {','.join(done)}")
            return True
        return False
    except Exception as e:
        if time.time() - LINK["err_logged"] > 300:
            LINK["err_logged"] = time.time()
            log(f"launcher publish error: {type(e).__name__}: {str(e)[:200]}")
        return False


def link_heartbeat():
    """renewer() se har 30s call hota hai; actual write LINK_MIN_INTERVAL pe throttle hoti hai."""
    if LINK.get("status") in ("online", "handoff", "starting"):
        publish_remote(LINK["status"])


def main():
    notify(f"▶️ [{NODE_NAME}] notebook started, installing.")
    setup()
    G["start_db"] = db("select now()", fetch=True)[0][0]
    deadline = time.time() + LEASE_WAIT_MAX
    while True:
        if shutdown_flag():
            log("shutdown requested before start")
            notify(f"🛑 [{NODE_NAME}] stopped due to shutdown (did not start).")
            return
        if try_acquire():
            break
        if time.time() > deadline:
            log("standby timeout: lease held by the other instance, exiting")
            notify(f"ℹ️ [{NODE_NAME}] was on standby (lease held by the other node), stopped.")
            return
        mark_waiter()
        time.sleep(10)
    log("lease acquired")
    threading.Thread(target=renewer, daemon=True).start()
    threading.Thread(target=tg_poller, daemon=True).start()
    try:
        db("update runtime_info set holder = %s, status = 'starting', launch_url = null, public_url = null, heartbeat_at = now() where id = 1", (NODE_NAME,))
        restore("dsh_home", DSH_HOME)
        restore("workspace", WORKSPACE)
        seed_resume_files()
        LAST_SIG["dsh_home"] = sig_of(DSH_HOME)
        LAST_SIG["workspace"] = sig_of(WORKSPACE)
        baseline_sent()
        threading.Thread(target=file_sender, daemon=True).start()
        start_stack()
        wait_public()
        publish("online")
        announce()

        restarts = []
        last_backup = time.time()
        last_check = time.time()
        last_flag = 0
        tfails = 0
        while True:
            time.sleep(5)
            up = time.time() - START
            if time.time() - last_flag > 10:
                last_flag = time.time()
                if shutdown_flag():
                    log("shutdown requested, stopping")
                    G["reason"] = "shutdown"
                    return
            if lost.is_set():
                log("lease lost, stopping")
                notify(f"⚠️ [{NODE_NAME}] lease lost, stopping.")
                return
            if up >= HARD_EXIT:
                log("hard exit time reached")
                notify(f"⏹ [{NODE_NAME}] approaching the 12h limit, stopping (peer was not ready: {not peer_ready()}).")
                return
            if up >= HANDOFF_AFTER and not H["started"]:
                H["started"] = True
                log("starting handoff to peer " + PEER_NAME)
                set_status("handoff")
                db("update runtime_info set handoff_at = now() where id = 1")
                notify(f"🔁 [{NODE_NAME}] handoff started, starting {PEER_NAME}.")
                threading.Thread(target=_handoff_job, daemon=True).start()
            if H["started"] and peer_ready():
                log("peer is ready, handing over")
                notify(f"🔁 [{NODE_NAME}] {PEER_NAME} is ready, handing over. {PEER_NAME} will send the new link.")
                return

            dead = S.dsh.poll() is not None or S.tunnel.poll() is not None
            if not dead and time.time() - last_check > 60:
                last_check = time.time()
                if http_alive(f"https://{S.host}/", 15, TUNNEL_DEAD):
                    tfails = 0
                else:
                    tfails += 1
                    log(f"tunnel check failed ({tfails}/3)")
                    if tfails >= 3:
                        dead = True
                if not dead and not http_alive(f"http://127.0.0.1:{PORT}/"):
                    log("local harness not responding")
                    dead = True
            if dead:
                now = time.time()
                restarts = [t for t in restarts if now - t < 600]
                if len(restarts) >= 3:
                    log("crash loop -> asking peer to take over")
                    notify(f"❌ [{NODE_NAME}] harness keeps crashing, handing over to {PEER_NAME}.")
                    trigger_peer()
                    end = time.time() + 900
                    while time.time() < end and not peer_ready():
                        time.sleep(10)
                    return
                restarts.append(now)
                tfails = 0
                log("restarting stack")
                stop_stack()
                start_stack()
                wait_public()
                publish("handoff" if H["started"] else "online")
                announce("(new link after restart)")
            if time.time() - last_backup > BACKUP_EVERY:
                backup_all()
                last_backup = time.time()
    finally:
        shutdown()


try:
    main()
except KeyboardInterrupt:
    log("interrupted")
except Exception as _e:
    notify(f"❌ [{NODE_NAME}] error: {type(_e).__name__}: {str(_e)[:300]}")
    raise
